In [2]:
import numpy as np
import cv2
from copy import copy
import itertools
import matplotlib.pyplot as plt
%matplotlib inline

import h3bot
from h3bot import Position
from h3bot.entity import Ship
from h3bot.positionals import get_position

from h3bot.astar_extended import astar_multiagent, astar_multiagent_no_cache, astar, dijkstras, greedy

In [3]:
np.random.seed(12345+12345)
size = 32
Position.width = Position.height = size

MOVE_COST_RATIO = 10
EXTRACT_RATIO = 4
MAX_HALITE = 1000

map = np.random.random_sample((size, size)) * MAX_HALITE

In [4]:
def neighbors(pos):
#     print(f"neighbors {pos}")
    for dx, dy in itertools.product([-1,0,1],[-1,0,1]):
        if not (dx and dy):
#             print(f"neighbors{pos}")
#             print(f"yield {pos.step+1}")
#             print(f"yield {pos+(dx,dy)}")
            yield Node(pos.step+1, (pos+(dx,dy)).norm())

def dist(pos, next):
#     print(f"dist{pos, next}")
    direct = get_position(pos) - next
    move = (min(direct.x, size-direct.x), min(direct.y, size-direct.y))
    return (move[0] + move[1])

def cost(pos, next):
#     print(f"cost{pos, next}")
    nvisits[0] += 1
    visited[next] += 0.2
    return (map[pos] / MOVE_COST_RATIO) * int(bool(pos!=next)) + dist(ship.position, home) * HOME_COST

def heur(pos, next):
#     print(f"heur{pos, next}")
    nvisits[0] += 1
    visited[next] += 0.2
#     if pos != next:
    return (map[next] * EXTRACT_RATIO) - dist(ship.position, home) * HOME_COST * MAX_HALITE
#     else:
#         return (map[next] * EXTRACT_RATIO) - dist(ship.position, home) * HOME_COST * MAX_HALITE

def draw_path(img, path):
    if path[0] is not None:
        for node in path:
            img[node] = 1.0
            visited[node] = 0

class Node(Position):
    def __new__(cls, step, x, y=None):
        if y is not None:
            x = (x,y)
        self = super().__new__(cls, x)
        return self
    
    def __init__(self, step, *args):
#         print(f"args {args}")
#         self.position = get_position(pos)
        super().__init__(*args)
        self.step = step
        print(self)
    
    def __repr__(self):
        return self.__class__.__name__ + '(' + str(self.step) + ', ' + super().__repr__()[1:-1] + ')'

    def __eq__(self, other):
#         return self.step == other.step and super().__eq__(self)
#         if isinstance(other, entity.Entity):
#             other = other.position
        return self.x == other[0] and self.y == other[1] and self.step == other.step
    
    def __hash__(self):
        return hash((self.x, self.y, self.step))

class SimShip(Ship):
    def __init__(self, pos, halite, step):
        super().__init__(0, 0, pos, halite)
        print(f"SimShip{pos, halite, step}")
        self.step = step

Node(0,1,2)==Node(0,1,2), Node(0, (1,2)) == Node(0, (1,2)), \
Node(0,1,2)==Node(1,1,2), Node(0, (1,2)) == Node(1, (1,2))

Node(0, 1x, 2y)
Node(0, 1x, 2y)
Node(0, 1x, 2y)
Node(0, 1x, 2y)
Node(0, 1x, 2y)
Node(1, 1x, 2y)
Node(0, 1x, 2y)
Node(1, 1x, 2y)


(True, True, False, False)

In [5]:
# ship = Position(np.random.randint(0, size), np.random.randint(0, size))
ship = SimShip(Node(0, (20,18)), 0, 0)
# home = (np.random.randint(0, size), np.random.randint(0, size))
home = Node(0,(1,2))

nvisits = [0]
HOME_COST = 1

Node(0, 20x, 18y)
SimShip(Node(0, 20x, 18y), 0, 0)
Node(0, 1x, 2y)


In [6]:
blank = np.zeros((size,size))
checks = np.zeros_like(map)
paths = np.zeros_like(map)
visited = np.zeros_like(map)
targets = np.zeros_like(map)
nvisits[0] = 0

start, end = ship.position, home
print(f"neighbors {list(neighbors(ship.position))}")

path = astar(start, goal=end,
    neighbors_fnct=neighbors,
    distance_between_fnct=cost,
    heuristic_cost_estimate_fnct=heur
)
draw_path(paths, path[start])
print(f"{len(path[start])} steps")
print(f"{len(set(path[start]))} unique nodes")
print(f"% visited {100*nvisits[0]/(size*size)}")

visited[start] = 1.0
visited[end] = 1.0
targets[start] = 1.0
targets[end] = 1.0

halite = 0
for n in path[start][1:]:
    halite += map[n] / EXTRACT_RATIO
    halite -= map[n] / MOVE_COST_RATIO
print(f"{halite} halite")

print(path[start])

plt.subplots(1,2, figsize=(15,15))

plt.subplot(1,2,1)
# rgb
image = np.dstack((visited, blank, paths))
plt.imshow(image)

plt.subplot(1,2,2)
# rgb
image = np.dstack((blank, map/MAX_HALITE, targets))
plt.imshow(image)

plt.show()

Node(1, 19x, 18y)
Node(1, 20x, 17y)
Node(1, 20x, 18y)
Node(1, 20x, 19y)
Node(1, 21x, 18y)
neighbors [Node(1, 19x, 18y), Node(1, 20x, 17y), Node(1, 20x, 18y), Node(1, 20x, 19y), Node(1, 21x, 18y)]
Node(1, 19x, 18y)
Node(1, 20x, 17y)
Node(1, 20x, 18y)
Node(1, 20x, 19y)
Node(1, 21x, 18y)
Node(2, 18x, 18y)
Node(2, 19x, 17y)
Node(2, 19x, 18y)
Node(2, 19x, 19y)
Node(2, 20x, 18y)
Node(2, 19x, 18y)
Node(2, 20x, 17y)
Node(2, 20x, 18y)
Node(2, 20x, 19y)
Node(2, 21x, 18y)
Node(2, 20x, 18y)
Node(2, 21x, 17y)
Node(2, 21x, 18y)
Node(2, 21x, 19y)
Node(2, 22x, 18y)
Node(2, 19x, 17y)
Node(2, 20x, 16y)
Node(2, 20x, 17y)
Node(2, 20x, 18y)
Node(2, 21x, 17y)
Node(2, 19x, 19y)
Node(2, 20x, 18y)
Node(2, 20x, 19y)
Node(2, 20x, 20y)
Node(2, 21x, 19y)
Node(3, 18x, 18y)
Node(3, 19x, 17y)
Node(3, 19x, 18y)
Node(3, 19x, 19y)
Node(3, 20x, 18y)
Node(3, 19x, 18y)
Node(3, 20x, 17y)
Node(3, 20x, 18y)
Node(3, 20x, 19y)
Node(3, 21x, 18y)
Node(3, 18x, 17y)
Node(3, 19x, 16y)
Node(3, 19x, 17y)
Node(3, 19x, 18y)
Node(3, 20x,

Node(6, 20x, 22y)
Node(6, 20x, 23y)
Node(6, 20x, 24y)
Node(6, 21x, 23y)
Node(6, 20x, 22y)
Node(6, 21x, 21y)
Node(6, 21x, 22y)
Node(6, 21x, 23y)
Node(6, 22x, 22y)
Node(6, 18x, 22y)
Node(6, 19x, 21y)
Node(6, 19x, 22y)
Node(6, 19x, 23y)
Node(6, 20x, 22y)
Node(7, 16x, 16y)
Node(7, 17x, 15y)
Node(7, 17x, 16y)
Node(7, 17x, 17y)
Node(7, 18x, 16y)
Node(7, 17x, 15y)
Node(7, 18x, 14y)
Node(7, 18x, 15y)
Node(7, 18x, 16y)
Node(7, 19x, 15y)
Node(7, 19x, 22y)
Node(7, 20x, 21y)
Node(7, 20x, 22y)
Node(7, 20x, 23y)
Node(7, 21x, 22y)
Node(7, 19x, 21y)
Node(7, 20x, 20y)
Node(7, 20x, 21y)
Node(7, 20x, 22y)
Node(7, 21x, 21y)
Node(7, 19x, 20y)
Node(7, 20x, 19y)
Node(7, 20x, 20y)
Node(7, 20x, 21y)
Node(7, 21x, 20y)
Node(8, 15x, 17y)
Node(8, 16x, 16y)
Node(8, 16x, 17y)
Node(8, 16x, 18y)
Node(8, 17x, 17y)
Node(6, 22x, 16y)
Node(6, 23x, 15y)
Node(6, 23x, 16y)
Node(6, 23x, 17y)
Node(6, 24x, 16y)
Node(7, 18x, 21y)
Node(7, 19x, 20y)
Node(7, 19x, 21y)
Node(7, 19x, 22y)
Node(7, 20x, 21y)
Node(6, 21x, 15y)
Node(6, 22

Node(8, 23x, 23y)
Node(12, 18x, 18y)
Node(12, 19x, 17y)
Node(12, 19x, 18y)
Node(12, 19x, 19y)
Node(12, 20x, 18y)
Node(8, 21x, 15y)
Node(8, 22x, 14y)
Node(8, 22x, 15y)
Node(8, 22x, 16y)
Node(8, 23x, 15y)
Node(12, 18x, 16y)
Node(12, 19x, 15y)
Node(12, 19x, 16y)
Node(12, 19x, 17y)
Node(12, 20x, 16y)
Node(12, 17x, 17y)
Node(12, 18x, 16y)
Node(12, 18x, 17y)
Node(12, 18x, 18y)
Node(12, 19x, 17y)
Node(12, 19x, 17y)
Node(12, 20x, 16y)
Node(12, 20x, 17y)
Node(12, 20x, 18y)
Node(12, 21x, 17y)
Node(7, 21x, 22y)
Node(7, 22x, 21y)
Node(7, 22x, 22y)
Node(7, 22x, 23y)
Node(7, 23x, 22y)
Node(10, 16x, 16y)
Node(10, 17x, 15y)
Node(10, 17x, 16y)
Node(10, 17x, 17y)
Node(10, 18x, 16y)
Node(10, 20x, 17y)
Node(10, 21x, 16y)
Node(10, 21x, 17y)
Node(10, 21x, 18y)
Node(10, 22x, 17y)
Node(8, 21x, 21y)
Node(8, 22x, 20y)
Node(8, 22x, 21y)
Node(8, 22x, 22y)
Node(8, 23x, 21y)
Node(10, 14x, 17y)
Node(10, 15x, 16y)
Node(10, 15x, 17y)
Node(10, 15x, 18y)
Node(10, 16x, 17y)
Node(10, 13x, 18y)
Node(10, 14x, 17y)
Node(10, 

Node(10, 21x, 25y)
Node(10, 21x, 26y)
Node(10, 21x, 27y)
Node(10, 22x, 26y)
Node(10, 20x, 25y)
Node(10, 21x, 24y)
Node(10, 21x, 25y)
Node(10, 21x, 26y)
Node(10, 22x, 25y)
Node(10, 21x, 25y)
Node(10, 22x, 24y)
Node(10, 22x, 25y)
Node(10, 22x, 26y)
Node(10, 23x, 25y)
Node(11, 15x, 22y)
Node(11, 16x, 21y)
Node(11, 16x, 22y)
Node(11, 16x, 23y)
Node(11, 17x, 22y)
Node(9, 19x, 13y)
Node(9, 20x, 12y)
Node(9, 20x, 13y)
Node(9, 20x, 14y)
Node(9, 21x, 13y)
Node(10, 21x, 15y)
Node(10, 22x, 14y)
Node(10, 22x, 15y)
Node(10, 22x, 16y)
Node(10, 23x, 15y)
Node(10, 19x, 26y)
Node(10, 20x, 25y)
Node(10, 20x, 26y)
Node(10, 20x, 27y)
Node(10, 21x, 26y)
Node(10, 18x, 25y)
Node(10, 19x, 24y)
Node(10, 19x, 25y)
Node(10, 19x, 26y)
Node(10, 20x, 25y)
Node(13, 15x, 17y)
Node(13, 16x, 16y)
Node(13, 16x, 17y)
Node(13, 16x, 18y)
Node(13, 17x, 17y)
Node(13, 15x, 16y)
Node(13, 16x, 15y)
Node(13, 16x, 16y)
Node(13, 16x, 17y)
Node(13, 17x, 16y)
Node(13, 14x, 16y)
Node(13, 15x, 15y)
Node(13, 15x, 16y)
Node(13, 15x, 17y

Node(12, 18x, 15y)
Node(12, 19x, 14y)
Node(13, 14x, 21y)
Node(13, 15x, 20y)
Node(13, 15x, 21y)
Node(13, 15x, 22y)
Node(13, 16x, 21y)
Node(12, 21x, 23y)
Node(12, 22x, 22y)
Node(12, 22x, 23y)
Node(12, 22x, 24y)
Node(12, 23x, 23y)
Node(11, 21x, 24y)
Node(11, 22x, 23y)
Node(11, 22x, 24y)
Node(11, 22x, 25y)
Node(11, 23x, 24y)
Node(11, 21x, 22y)
Node(11, 22x, 21y)
Node(11, 22x, 22y)
Node(11, 22x, 23y)
Node(11, 23x, 22y)
Node(12, 20x, 15y)
Node(12, 21x, 14y)
Node(12, 21x, 15y)
Node(12, 21x, 16y)
Node(12, 22x, 15y)
Node(12, 16x, 24y)
Node(12, 17x, 23y)
Node(12, 17x, 24y)
Node(12, 17x, 25y)
Node(12, 18x, 24y)
Node(12, 15x, 23y)
Node(12, 16x, 22y)
Node(12, 16x, 23y)
Node(12, 16x, 24y)
Node(12, 17x, 23y)
Node(15, 20x, 18y)
Node(15, 21x, 17y)
Node(15, 21x, 18y)
Node(15, 21x, 19y)
Node(15, 22x, 18y)
Node(12, 21x, 21y)
Node(12, 22x, 20y)
Node(12, 22x, 21y)
Node(12, 22x, 22y)
Node(12, 23x, 21y)
Node(11, 19x, 28y)
Node(11, 20x, 27y)
Node(11, 20x, 28y)
Node(11, 20x, 29y)
Node(11, 21x, 28y)
Node(11, 18x

...

Node(26, 27x, 15y)
Node(26, 28x, 14y)
Node(26, 28x, 15y)
Node(26, 28x, 16y)
Node(26, 29x, 15y)
Node(22, 14x, 29y)
Node(22, 15x, 28y)
Node(22, 15x, 29y)
Node(22, 15x, 30y)
Node(22, 16x, 29y)
Node(24, 7x, 19y)
Node(24, 8x, 18y)
Node(24, 8x, 19y)
Node(24, 8x, 20y)
Node(24, 9x, 19y)
Node(32, 18x, 18y)
Node(32, 19x, 17y)
Node(32, 19x, 18y)
Node(32, 19x, 19y)
Node(32, 20x, 18y)
Node(32, 19x, 18y)
Node(32, 20x, 17y)
Node(32, 20x, 18y)
Node(32, 20x, 19y)
Node(32, 21x, 18y)
Node(32, 18x, 17y)
Node(32, 19x, 16y)
Node(32, 19x, 17y)
Node(32, 19x, 18y)
Node(32, 20x, 17y)
Node(32, 14x, 21y)
Node(32, 15x, 20y)
Node(32, 15x, 21y)
Node(32, 15x, 22y)
Node(32, 16x, 21y)
Node(29, 22x, 24y)
Node(29, 23x, 23y)
Node(29, 23x, 24y)
Node(29, 23x, 25y)
Node(29, 24x, 24y)
Node(28, 24x, 15y)
Node(28, 25x, 14y)
Node(28, 25x, 15y)
Node(28, 25x, 16y)
Node(28, 26x, 15y)
Node(28, 8x, 23y)
Node(28, 9x, 22y)
Node(28, 9x, 23y)
Node(28, 9x, 24y)
Node(28, 10x, 23y)
Node(28, 9x, 24y)
Node(28, 10x, 23y)
Node(28, 10x, 24y)
Nod

Node(32, 12x, 21y)
Node(32, 11x, 22y)
Node(32, 12x, 21y)
Node(32, 12x, 22y)
Node(32, 12x, 23y)
Node(32, 13x, 22y)
Node(32, 12x, 21y)
Node(32, 13x, 20y)
Node(32, 13x, 21y)
Node(32, 13x, 22y)
Node(32, 14x, 21y)
Node(32, 11x, 21y)
Node(32, 12x, 20y)
Node(32, 12x, 21y)
Node(32, 12x, 22y)
Node(32, 13x, 21y)
Node(32, 23x, 18y)
Node(32, 24x, 17y)
Node(32, 24x, 18y)
Node(32, 24x, 19y)
Node(32, 25x, 18y)
Node(31, 11x, 17y)
Node(31, 12x, 16y)
Node(31, 12x, 17y)
Node(31, 12x, 18y)
Node(31, 13x, 17y)
Node(31, 10x, 16y)
Node(31, 11x, 15y)
Node(31, 11x, 16y)
Node(31, 11x, 17y)
Node(31, 12x, 16y)
Node(19, 1x, 14y)
Node(19, 2x, 13y)
Node(19, 2x, 14y)
Node(19, 2x, 15y)
Node(19, 3x, 14y)
Node(19, 0x, 13y)
Node(19, 1x, 12y)
Node(19, 1x, 13y)
Node(19, 1x, 14y)
Node(19, 2x, 13y)
Node(24, 30x, 24y)
Node(24, 31x, 23y)
Node(24, 31x, 24y)
Node(24, 31x, 25y)
Node(24, 0x, 24y)
Node(22, 20x, 5y)
Node(22, 21x, 4y)
Node(22, 21x, 5y)
Node(22, 21x, 6y)
Node(22, 22x, 5y)
Node(24, 29x, 25y)
Node(24, 30x, 24y)
Node(24, 

Node(33, 17x, 21y)
Node(33, 18x, 20y)
Node(33, 18x, 21y)
Node(33, 18x, 22y)
Node(33, 19x, 21y)
Node(33, 17x, 20y)
Node(33, 18x, 19y)
Node(33, 18x, 20y)
Node(33, 18x, 21y)
Node(33, 19x, 20y)
Node(27, 11x, 29y)
Node(27, 12x, 28y)
Node(27, 12x, 29y)
Node(27, 12x, 30y)
Node(27, 13x, 29y)
Node(27, 12x, 28y)
Node(27, 13x, 27y)
Node(27, 13x, 28y)
Node(27, 13x, 29y)
Node(27, 14x, 28y)
Node(24, 24x, 10y)
Node(24, 25x, 9y)
Node(24, 25x, 10y)
Node(24, 25x, 11y)
Node(24, 26x, 10y)
Node(27, 7x, 26y)
Node(27, 8x, 25y)
Node(27, 8x, 26y)
Node(27, 8x, 27y)
Node(27, 9x, 26y)
Node(27, 8x, 25y)
Node(27, 9x, 24y)
Node(27, 9x, 25y)
Node(27, 9x, 26y)
Node(27, 10x, 25y)
Node(27, 8x, 27y)
Node(27, 9x, 26y)
Node(27, 9x, 27y)
Node(27, 9x, 28y)
Node(27, 10x, 27y)
Node(27, 19x, 30y)
Node(27, 20x, 29y)
Node(27, 20x, 30y)
Node(27, 20x, 31y)
Node(27, 21x, 30y)
Node(23, 6x, 23y)
Node(23, 7x, 22y)
Node(23, 7x, 23y)
Node(23, 7x, 24y)
Node(23, 8x, 23y)
Node(28, 25x, 15y)
Node(28, 26x, 14y)
Node(28, 26x, 15y)
Node(28, 26x

Node(27, 22x, 11y)
Node(27, 23x, 10y)
Node(27, 23x, 11y)
Node(27, 23x, 12y)
Node(27, 24x, 11y)
Node(27, 14x, 26y)
Node(27, 15x, 25y)
Node(27, 15x, 26y)
Node(27, 15x, 27y)
Node(27, 16x, 26y)
Node(24, 18x, 6y)
Node(24, 19x, 5y)
Node(24, 19x, 6y)
Node(24, 19x, 7y)
Node(24, 20x, 6y)
Node(29, 17x, 8y)
Node(29, 18x, 7y)
Node(29, 18x, 8y)
Node(29, 18x, 9y)
Node(29, 19x, 8y)
Node(30, 12x, 12y)
Node(30, 13x, 11y)
Node(30, 13x, 12y)
Node(30, 13x, 13y)
Node(30, 14x, 12y)
Node(21, 1x, 18y)
Node(21, 2x, 17y)
Node(21, 2x, 18y)
Node(21, 2x, 19y)
Node(21, 3x, 18y)
Node(30, 17x, 14y)
Node(30, 18x, 13y)
Node(30, 18x, 14y)
Node(30, 18x, 15y)
Node(30, 19x, 14y)
Node(31, 20x, 13y)
Node(31, 21x, 12y)
Node(31, 21x, 13y)
Node(31, 21x, 14y)
Node(31, 22x, 13y)
Node(20, 26x, 8y)
Node(20, 27x, 7y)
Node(20, 27x, 8y)
Node(20, 27x, 9y)
Node(20, 28x, 8y)
Node(20, 27x, 9y)
Node(20, 28x, 8y)
Node(20, 28x, 9y)
Node(20, 28x, 10y)
Node(20, 29x, 9y)
Node(28, 17x, 27y)
Node(28, 18x, 26y)
Node(28, 18x, 27y)
Node(28, 18x, 28y

KeyboardInterrupt: 